# Imports

In [15]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split

from feature_engine.selection import DropFeatures

## Utils

In [2]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet')

In [4]:
X_train.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
0,0.50,0.203815,0.328482,0.154734,1.585901,-0.308849,1.487008,-1.486487,1.666667,1.040769,...,0.947453,0.005655,0.020178,0.000549,0.997580,0.001872,0.012740,0.40,0,0.781831
1,0.50,0.199575,0.327181,0.176114,0.229628,-1.066602,0.033534,1.440545,-1.000000,-5.009333,...,0.074107,0.081854,0.098684,0.015175,0.007697,0.977128,0.013316,0.20,1,0.885456
2,0.50,0.218840,0.326837,0.187526,2.116616,0.638343,1.902200,-1.486487,1.000000,-1.970285,...,0.226475,0.059038,0.614716,0.024077,0.912645,0.063278,0.014095,0.65,0,0.537300
3,0.25,0.200526,0.100888,0.145166,-1.244581,-0.498287,-1.029455,-0.510810,0.000000,0.338641,...,0.088099,0.041344,0.113803,0.950726,0.010153,0.039121,0.010598,0.35,0,0.239316
4,0.50,0.192082,0.327657,0.214557,0.170660,-1.445478,0.092589,-1.486487,1.000000,0.169816,...,0.256086,0.040890,0.083385,0.009589,0.004844,0.985566,0.009270,0.10,1,0.906308


In [5]:
X_test.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
439140,0.25,0.182308,0.101132,0.133462,-0.124182,-1.066602,0.261317,-0.510810,0.000000,0.396609,...,0.001144,0.000465,0.000894,0.067702,0.028176,0.904122,0.010708,0.20,0,0.954721
439141,0.25,0.356316,0.101132,0.150547,0.052723,-1.634917,0.300590,-0.510810,0.000000,0.470778,...,0.081596,0.033933,0.053686,0.019477,0.011500,0.969023,0.011005,0.05,0,0.963550
439142,0.25,0.121796,0.101132,0.133462,0.052723,0.259466,0.489100,-0.510810,0.000000,0.301036,...,0.087683,0.072629,0.701444,0.038897,0.031448,0.929655,0.010768,0.55,0,0.992709
439143,0.00,0.198847,0.193475,0.253713,-1.008708,1.017219,-1.025511,0.464868,0.333333,0.724449,...,0.023113,0.022859,0.935769,0.984826,0.002887,0.012287,0.010530,0.75,0,0.242362
439144,0.50,0.197127,0.327536,0.114051,1.703838,0.448905,1.518343,0.464868,2.333333,0.003617,...,0.718809,0.040632,0.151701,0.002043,0.991340,0.006618,0.010090,0.60,0,0.766044


# Machine Learning

## Base

In [6]:
cv_results = cross_val_score(
    estimator=HistGradientBoostingClassifier(class_weight='balanced', verbose=0), 
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1
)

In [7]:
cv_results.mean()

np.float64(0.9429598711008129)

In [9]:
model = HistGradientBoostingClassifier(class_weight='balanced', verbose=0)
model.fit(X_train, y_train.PitNextLap)

HistGradientBoostingClassifier(class_weight='balanced')

## Feature Selection

In [10]:
perm_result = permutation_importance(
    estimator=model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1, 
    scoring='roc_auc'
)


importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [11]:
importance_df

,feature,importance_mean,importance_std
12,robust_scaler__stint,0.098400,4.046051e-04
7,standard_scaler__year,0.089156,6.239892e-04
18,robust_scaler__tyre_life_ratio,0.025835,1.152948e-04
14,robust_scaler__tyrelife,0.018556,1.312438e-04
3,target_encoder__race,0.009441,6.613894e-05
10,robust_scaler__laptime_delta,0.005352,5.615044e-05
15,robust_scaler__delta_x_tyre_life,0.003745,1.218361e-04
0,ordinal_encoder__compound,0.002248,3.385269e-05
8,robust_scaler__position_change,0.001971,1.753807e-05
22,remainder__pitstop,0.001925,5.760198e-05


In [12]:
importance_df.query("importance_mean <= 0").feature.tolist()

['remainder__raceprogress_high',
 'remainder__lapnumber_high',
 'remainder__position_high',
 'remainder__laptime_s_high',
 'remainder__position_norm',
 'remainder__stint_high',
 'remainder__is_pit_lap',
 'remainder__lapnumber_low']

In [16]:
drop_feat = DropFeatures(features_to_drop=importance_df.query("importance_mean <= 0").feature.tolist())

## Fine Tuning

In [18]:
def objective(trial, X, y):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 20, 100),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 10.0),
        "max_bins": trial.suggest_int("max_bins", 64, 255),
        "max_iter": 1000,
        "early_stopping": True,
        "validation_fraction": 0.1,
        "n_iter_no_change": 50,
        "random_state": 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = Pipeline([('drop_feat', drop_feat), ('hist', HistGradientBoostingClassifier(**params))])
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, n_jobs=-1, show_progress_bar=True)


print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-05 18:43:36,045] A new study created in memory with name: no-name-42647aae-1ced-4ced-af24-74def7f7a378


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-05 18:52:39,380] Trial 1 finished with value: 0.9381805118892965 and parameters: {'learning_rate': 0.028091270865478902, 'max_depth': 3, 'min_samples_leaf': 26, 'l2_regularization': 4.543837205389343, 'max_bins': 151}. Best is trial 1 with value: 0.9381805118892965.
[I 2026-05-05 18:52:44,086] Trial 4 finished with value: 0.9373857886826906 and parameters: {'learning_rate': 0.02189422965538291, 'max_depth': 3, 'min_samples_leaf': 37, 'l2_regularization': 1.4108161263859986, 'max_bins': 130}. Best is trial 1 with value: 0.9381805118892965.
[I 2026-05-05 18:54:45,079] Trial 5 finished with value: 0.9442576086486953 and parameters: {'learning_rate': 0.04984954840138133, 'max_depth': 4, 'min_samples_leaf': 37, 'l2_regularization': 0.4151877509876112, 'max_bins': 245}. Best is trial 5 with value: 0.9442576086486953.
[I 2026-05-05 18:54:53,353] Trial 2 finished with value: 0.9469034106017903 and parameters: {'learning_rate': 0.14907104037881883, 'max_depth': 8, 'min_samples_leaf':

In [19]:
model_tuned = HistGradientBoostingClassifier(
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50,
    random_state=42,
    **study.best_params
)

pipe_tuned = Pipeline([('drop_feat', drop_feat), ('hist', model_tuned)])
pipe_tuned.fit(X_train, y_train.PitNextLap)

Pipeline(steps=[('drop_feat',
                 DropFeatures(features_to_drop=['remainder__raceprogress_high',
                                                'remainder__lapnumber_high',
                                                'remainder__position_high',
                                                'remainder__laptime_s_high',
                                                'remainder__position_norm',
                                                'remainder__stint_high',
                                                'remainder__is_pit_lap',
                                                'remainder__lapnumber_low'])),
                ('hist',
                 HistGradientBoostingClassifier(early_stopping=True,
                                                l2_regularization=6.70145048510531,
                                                learning_rate=0.07348770421656471,
                                                max_bins=190, max_depth=10,
                                                max_iter=1000,
                                                min_samples_leaf=81,
                                                n_iter_no_change=50,
                                                random_state=42))])

# Deploy

In [20]:
dump_pickle(pipe_tuned, '../models/model_histgradientboosting.pkl')